# METADATA PREPARATION

In [ ]:
import pandas as pd
import os

INPUT_FOLDER = 'fma_metadata'
OUTPUT_FOLDER = 'processed'

# Create output folder if it doesn't exist
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)
    print(f"Directory '{OUTPUT_FOLDER}' created.")

---

## 1. GENRES

### 1.1. Metadata Preparation

In [9]:
# --- 1. LOAD DATA ---
# Load raw genres to get metadata like colors and handles
raw_genres = pd.read_csv(f'{INPUT_FOLDER}/raw_genres.csv')

# Load the curated genres file just to get the track counts
tracks_count = pd.read_csv(f'{INPUT_FOLDER}/genres.csv')[['genre_id', '#tracks']]

# --- 2. MERGE AND CLEANING ---
# Merge the two dataframes on 'genre_id'
# 'left': keep all genres from raw, add info from tracks_count where it matches
df_final = pd.merge(raw_genres, tracks_count, on='genre_id', how='left')

# Rename the column #tracks to be more descriptive
df_final.rename(columns={'#tracks': 'genre_n_tracks'}, inplace=True)

# Fill NaN values in parent_id with 0 (root) and convert to integer
df_final['genre_parent_id'] = df_final['genre_parent_id'].fillna(0).astype(int)

# Ensure track count is also integer 
df_final['genre_n_tracks'] = df_final['genre_n_tracks'].fillna(0).astype(int)

# Select and reorder the desired columns
desired_columns = [
    'genre_id', 
    'genre_handle', 
    'genre_title', 
    'genre_parent_id', 
    'genre_n_tracks', 
    'genre_color'
]

# Apply column filter
df_final = df_final[desired_columns]

# --- 3. EXPORT ---
output_path = f'{OUTPUT_FOLDER}/genres.csv'
df_final.to_csv(output_path, index=False)
print(f"File successfully saved to: {output_path}")

File successfully saved to: ../data/processed/genres.csv


### 1.2. Visualization

In [4]:
display(df_final)

,genre_id,genre_handle,genre_title,genre_parent_id,genre_n_tracks,genre_color
0,1,Avant-Garde,Avant-Garde,38,8693,#006666
1,2,International,International,0,5271,#CC3300
2,3,Blues,Blues,0,1752,#000099
3,4,Jazz,Jazz,0,4126,#990099
4,5,Classical,Classical,0,4106,#8A8A65
...,...,...,...,...,...,...
159,1032,Turkish,Turkish,102,60,#CC3300
160,1060,tango,Tango,46,30,#CC3300
161,1156,Fado,Fado,130,26,#CC3300
162,1193,Christmas,Christmas,763,72,#D4A017


### 1.3. Tree Visualization

In [10]:
print("\n--- Genre Tree Hierarchy ---")

def print_genre_tree(df, parent_id=0, level=0):
    """
    Recursive function to print the genre tree.
    Assumes root nodes have parent_id == 0.
    """
    # Find children of the current parent_id
    children = df[df['genre_parent_id'] == parent_id]
    
    # If using a subset, some parents might point to IDs that don't exist in the list.
    # We start searching for Roots (0).
    
    for _, row in children.iterrows():
        # Visual indentation based on depth level
        indent = "    " * level + "|-- " if level > 0 else ""
        
        print(f"{indent}{row['genre_title']} (ID: {row['genre_id']}) - Tracks: {row['genre_n_tracks']}")
        
        # Recursion: find children of this current genre
        print_genre_tree(df, row['genre_id'], level + 1)

# Start the tree from Root (ID 0) 
print_genre_tree(df_final, parent_id=0)


--- Genre Tree Hierarchy ---
International (ID: 2) - Tracks: 5271
    |-- Latin America (ID: 46) - Tracks: 573
        |-- Cumbia (ID: 502) - Tracks: 67
        |-- Salsa (ID: 808) - Tracks: 12
        |-- Tango (ID: 1060) - Tracks: 30
    |-- French (ID: 77) - Tracks: 387
    |-- Reggae - Dub (ID: 79) - Tracks: 880
        |-- Reggae - Dancehall (ID: 602) - Tracks: 94
    |-- Indian (ID: 86) - Tracks: 216
        |-- N. Indian Traditional (ID: 173) - Tracks: 4
        |-- South Indian Traditional (ID: 174) - Tracks: 17
        |-- Bollywood (ID: 175) - Tracks: 0
    |-- African (ID: 92) - Tracks: 329
        |-- Afrobeat (ID: 81) - Tracks: 110
        |-- North African (ID: 214) - Tracks: 40
    |-- Middle East (ID: 102) - Tracks: 176
        |-- Turkish (ID: 1032) - Tracks: 60
    |-- Polka (ID: 117) - Tracks: 62
    |-- Balkan (ID: 118) - Tracks: 608
    |-- Europe (ID: 130) - Tracks: 727
        |-- Romany (Gypsy) (ID: 524) - Tracks: 112
        |-- Spanish (ID: 619) - Tracks: 115

## 
---

## 2. ARTISTS

### 2.1. Metadata Preparation

In [11]:
# --- 1. LOAD DATA ---
# Load raw artists csv
raw_artists = pd.read_csv(f'{INPUT_FOLDER}/raw_artists.csv')

print(f"Original columns: {list(raw_artists.columns)}")

# --- 2. SELECT RELEVANT COLUMNS ---
# We select only the columns that are useful for metadata analysis and visualization
columns_to_keep = [
    'artist_id',
    'artist_name',
    'artist_location',
    'artist_latitude',
    'artist_longitude',
    'artist_active_year_begin',
    'artist_active_year_end'
]

# Create a new clean dataframe
artists_clean = raw_artists[columns_to_keep].copy()

# --- 3. DATA CLEANING ---
# Set artist_id as index
artists_clean.set_index('artist_id', inplace=True)

# Sort by ID
artists_clean.sort_index(inplace=True)

# --- 4. EXPORT ---
output_path = f'{OUTPUT_FOLDER}/artists.csv'
artists_clean.to_csv(output_path)
print(f"Cleaned artists file saved to: {output_path}")

Original columns: ['artist_id', 'artist_active_year_begin', 'artist_active_year_end', 'artist_associated_labels', 'artist_bio', 'artist_comments', 'artist_contact', 'artist_date_created', 'artist_donation_url', 'artist_favorites', 'artist_flattr_name', 'artist_handle', 'artist_image_file', 'artist_images', 'artist_latitude', 'artist_location', 'artist_longitude', 'artist_members', 'artist_name', 'artist_paypal_name', 'artist_related_projects', 'artist_url', 'artist_website', 'artist_wikipedia_page', 'tags']
Cleaned artists file saved to: ../data/processed/artists.csv


### 2.2. Visualization

In [12]:
display(artists_clean)

,artist_name,artist_location,artist_latitude,artist_longitude,artist_active_year_begin,artist_active_year_end
artist_id,,,,,,
1,AWOL,New Jersey,40.058324,-74.405661,2006.0,NaN
2,Free Matter for the Blind,NaN,NaN,NaN,NaN,NaN
3,Clover,NaN,NaN,NaN,NaN,NaN
4,Nicky Cook,Colchester England,51.895927,0.891874,1990.0,2011.0
5,JFrank,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
24352,Alex Mason/The Minor Emotion,NaN,NaN,NaN,NaN,NaN
24353,Alex Mason/Artem Gribov,NaN,NaN,NaN,NaN,NaN
24354,Alex Mason/BlackSunAeon Music,NaN,NaN,NaN,NaN,NaN


## 
---

## 3. ALBUMS

### 3.1. Metadata Preparation

In [6]:
# --- 1. LOAD DATA ---
# Load raw albums csv
raw_albums = pd.read_csv(f'{INPUT_FOLDER}/raw_albums.csv')

print(f"Original columns: {list(raw_albums.columns)}")

# --- 2. SELECT RELEVANT COLUMNS ---
# Keep structural info (ID, Title, Type) and temporal info (Dates)
columns_to_keep = [
    'album_id',
    'album_title',
    'album_type',
    'album_date_released',
    'album_date_created',
    'artist_name'
]

# Create a clean dataframe
albums_clean = raw_albums[columns_to_keep].copy()

# --- 3. DATA CLEANING ---
# Set album_id as index
albums_clean.set_index('album_id', inplace=True)

# Sort by ID for faster lookups later
albums_clean.sort_index(inplace=True)

# Handle Missing Dates: 
# If 'date_released' is missing, fallback to 'date_created' (better than nothing for timeline analysis)
albums_clean['album_date_released'] = albums_clean['album_date_released'].fillna(albums_clean['album_date_created'])

# Fill NaN with 'Unknown'
albums_clean['album_type'] = albums_clean['album_type'].fillna('Unknown')

# --- 4. EXPORT ---
output_path = f'{OUTPUT_FOLDER}/albums.csv'
albums_clean.to_csv(output_path)
print(f"Cleaned albums file saved to: {output_path}")

Original columns: ['album_id', 'album_comments', 'album_date_created', 'album_date_released', 'album_engineer', 'album_favorites', 'album_handle', 'album_image_file', 'album_images', 'album_information', 'album_listens', 'album_producer', 'album_title', 'album_tracks', 'album_type', 'album_url', 'artist_name', 'artist_url', 'tags']
Cleaned albums file saved to: ../data/processed/albums.csv


### 3.2. Visualization

In [7]:
display(albums_clean)

,album_title,album_type,album_date_released,album_date_created,artist_name
album_id,,,,,
1,AWOL - A Way Of Life,Album,1/05/2009,11/26/2008 01:44:45 AM,AWOL
4,Niris,Album,1/06/2009,11/26/2008 01:45:05 AM,Nicky Cook
6,Constant Hitmaker,Album,2/06/2008,11/26/2008 01:45:08 AM,Kurt Vile
58,mp3,Single Tracks,1/07/2009,11/26/2008 01:49:19 AM,Abominog
59,Live at LACE,Live Performance,12/01/2006,11/26/2008 01:49:35 AM,Airway
...,...,...,...,...,...
22932,Words Or Silence 2,Unknown,3/30/2017 08:31:12 AM,3/30/2017 08:31:12 AM,P C III
22935,Return,Album,3/30/2017,3/30/2017 09:50:32 AM,Alex Mason
22936,Scissors Paper Stone,Album,3/30/2017 10:45:13 AM,3/30/2017 10:45:13 AM,Greg Atkinson


## 
---

## 4. TRACKS

### 4.1. Metadata Preparation

In [8]:
# --- 1. LOAD DATA ---

# A. Load RAW Tracks (Contains technical info and IDs)
raw_tracks = pd.read_csv(f'{INPUT_FOLDER}/raw_tracks.csv')

# B. Load CURATED Tracks (Contains 'subset' and 'genre_top')
tracks_curated = pd.read_csv(f'{INPUT_FOLDER}/tracks.csv', index_col=0, header=[0, 1])

# --- 2. PREPARE CURATED DATA FOR MERGE ---
# We only want specific columns from the curated file:
# ('set', 'subset') -> Tells us if it is Small/Medium/Large
# ('track', 'genre_top') -> The main genre label
# ('track', 'genres') -> All of the genres of the track
curated_subset = tracks_curated[[
    ('set', 'subset'), 
    ('track', 'genre_top'), 
    ('track', 'genres')
]].copy()

# Flatten the columns (remove the hierarchy) to make it simple: 'subset', 'genre_top'
curated_subset.columns = ['subset', 'genre_top', 'genres']

# --- 3. MERGE ---
# raw_tracks has 'track_id' as a column, curated_subset has it as index.
# We join them on track_id.
raw_tracks.set_index('track_id', inplace=True)
tracks_full = raw_tracks.join(curated_subset, how='inner') # inner join ensures we only keep tracks present in both

print(f"Original columns: {list(tracks_full.columns)}")

# --- 4. SELECT RELEVANT COLUMNS ---
# We keep structural IDs, duration, titles, and the crucial classification labels
columns_to_keep = [
    'track_title',
    'artist_id',
    'album_id',
    'track_duration',
    'track_bit_rate',
    'subset', 
    'genre_top',
    'genres'
]

# Create clean dataframe
tracks_clean = tracks_full[columns_to_keep].copy()

# --- 5. DATA CLEANING ---
# Sort by ID
tracks_clean.sort_index(inplace=True)

# --- 6. EXPORT ---
output_path = f'{OUTPUT_FOLDER}/tracks.csv'
tracks_clean.to_csv(output_path)
print(f"Cleaned tracks file saved to: {output_path}")

Original columns: ['album_id', 'album_title', 'album_url', 'artist_id', 'artist_name', 'artist_url', 'artist_website', 'license_image_file', 'license_image_file_large', 'license_parent_id', 'license_title', 'license_url', 'tags', 'track_bit_rate', 'track_comments', 'track_composer', 'track_copyright_c', 'track_copyright_p', 'track_date_created', 'track_date_recorded', 'track_disc_number', 'track_duration', 'track_explicit', 'track_explicit_notes', 'track_favorites', 'track_file', 'track_genres', 'track_image_file', 'track_information', 'track_instrumental', 'track_interest', 'track_language_code', 'track_listens', 'track_lyricist', 'track_number', 'track_publisher', 'track_title', 'track_url', 'subset', 'genre_top', 'genres']
Cleaned tracks file saved to: ../data/processed/tracks.csv


### 4.2. Visualization

In [5]:
display(tracks_clean)

,track_title,artist_id,album_id,track_duration,track_date_recorded,track_date_created,track_bit_rate,subset,genre_top,genres
track_id,,,,,,,,,,
2,Food,1,1.0,02:48,11/26/2008,11/26/2008 01:48:12 AM,256000.0,small,Hip-Hop,[21]
3,Electric Ave,1,1.0,03:57,11/26/2008,11/26/2008 01:48:14 AM,256000.0,medium,Hip-Hop,[21]
5,This World,1,1.0,03:26,11/26/2008,11/26/2008 01:48:20 AM,256000.0,small,Hip-Hop,[21]
10,Freeway,6,6.0,02:41,11/26/2008,11/25/2008 05:49:06 PM,192000.0,small,Pop,[10]
20,Spiritual Level,4,4.0,05:11,1/01/2008,11/26/2008 01:48:56 AM,256000.0,large,NaN,"[76, 103]"
...,...,...,...,...,...,...,...,...,...,...
155316,The Auger,24357,22940.0,02:42,NaN,3/30/2017 03:23:34 PM,320000.0,large,Rock,[25]
155317,Let's Skin Ruby,24357,22940.0,03:37,NaN,3/30/2017 03:23:36 PM,320000.0,large,Rock,[25]
155318,My House Smells Like Kim Deal/Pulp,24357,22940.0,06:44,NaN,3/30/2017 03:23:37 PM,320000.0,large,Rock,[25]


## 
---